# 07a · Gold — Ingresantes (Dimensiones + Hecho)

**Objetivo:** construir **solo la parte de Ingresantes** de la capa Gold: las **5 dimensiones** (se escriben en `data/Gold/` para que el notebook 07b las recargue) y **`fact_ingresantes.parquet`**.

**Alcance:** NO se procesa Matriculados aquí (hecho de Matriculados → notebook 07b). Al final se **libera toda la memoria** (`del` + `gc.collect()`).

**Entradas:** `data/Silver/ingresantes_clean_v2.parquet`, `data/Silver/matriculados_clean_v2.parquet` (solo para catálogos de dimensiones).

**Salidas:** `data/Gold/dim_*.parquet` (5 tablas) y `data/Gold/fact_ingresantes.parquet`.

In [1]:
# Configuración: límite de hilos ANTES de importar Polars (12 núcleos)
import os
os.environ['POLARS_MAX_THREADS'] = '12'

import gc
import json
from datetime import datetime
from pathlib import Path

import polars as pl

pl.Config.set_streaming_chunk_size(32 * 1024 * 1024)  # 32 MB por lote de streaming

print('polars', pl.__version__)
print('hilos activos:', pl.thread_pool_size())


def rss_actual_gb():
    '''RSS actual del proceso en GB (Linux, /proc/self/statm).'''
    try:
        with open('/proc/self/statm', encoding='utf-8') as fh:
            paginas = int(fh.read().split()[1])
        return paginas * os.sysconf('SC_PAGE_SIZE') / (1024**3)
    except (OSError, ValueError, IndexError):
        return float('nan')


print(f'RSS inicial: {rss_actual_gb():.2f} GB')
print()

# Rutas del proyecto (misma detección automática que los notebooks 01-07)
current_dir = Path.cwd()
if (current_dir / 'data').exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / 'data').exists():
    PROJECT_ROOT = current_dir.parent
else:
    raise FileNotFoundError('No se encontró la carpeta data. Ejecuta desde la raíz o desde notebooks/.')

SILVER = PROJECT_ROOT / 'data' / 'Silver'
GOLD = PROJECT_ROOT / 'data' / 'Gold'
SCHEMAS = PROJECT_ROOT / 'data' / 'schemas'
ING_V2 = SILVER / 'ingresantes_clean_v2.parquet'
MAT_V2 = SILVER / 'matriculados_clean_v2.parquet'

assert ING_V2.exists(), f'No existe {ING_V2.name}. Ejecuta primero 04_limpieza_ingresantes.ipynb.'
assert MAT_V2.exists(), f'No existe {MAT_V2.name}. Ejecuta primero 04_limpieza_matriculados.ipynb.'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('Silver:', SILVER)
print('Gold:', GOLD)
print('Schemas:', SCHEMAS)

polars 1.44.1
hilos activos: 12
RSS inicial: 0.08 GB

PROJECT_ROOT: /mnt/datos/Proyectos/A.Prueba Tecnica UCSP
Silver: /mnt/datos/Proyectos/A.Prueba Tecnica UCSP/data/Silver
Gold: /mnt/datos/Proyectos/A.Prueba Tecnica UCSP/data/Gold
Schemas: /mnt/datos/Proyectos/A.Prueba Tecnica UCSP/data/schemas


---
## SECCIÓN 1 · Construcción de dimensiones Gold (5 tablas, con SK)

Catálogos pequeños vía `scan_parquet` + `unique()` + `collect()` (sin OOM); se añade `SK_*` con `with_row_index(offset=1)` y se guardan en `data/Gold/` para que el notebook **07b** (Matriculados) los recargue desde disco.

> Matriculados se usa aquí **solo** para extraer los catálogos de las dimensiones conformadas y `DimLocal`; NO se construye su hecho.

In [2]:
GOLD.mkdir(exist_ok=True)
DEPARTAMENTOS_SUR = ['AREQUIPA', 'CUSCO', 'TACNA', 'PUNO', 'MOQUEGUA', 'APURIMAC']

# --- DimUniversidad: unión ING (LICENCIADO) + MAT (LICENCIA) → ESTADO_LICENCIAMIENTO ---
dim_univ = (
    pl.concat([
        pl.scan_parquet(ING_V2)
        .select(['CODIGO_INEI', 'NOMBRE_ENTIDAD', 'TIPO_ENTIDAD', 'TIPO_GESTION', 'LICENCIADO', 'TIPO_CONSTITUCION'])
        .rename({'LICENCIADO': 'LICENCIA'})
        .unique()
        .collect(),
        pl.scan_parquet(MAT_V2)
        .select(['CODIGO_INEI', 'NOMBRE_ENTIDAD', 'TIPO_ENTIDAD', 'TIPO_GESTION', 'LICENCIA', 'TIPO_CONSTITUCION'])
        .unique()
        .collect(),
    ])
    .unique()
    .unique(subset=['CODIGO_INEI'])
    .sort('CODIGO_INEI')
    .with_columns(
        pl.when(pl.col('LICENCIA') == 'LICENCIADA').then(pl.lit('LICENCIADO'))
        .when(pl.col('LICENCIA') == 'LEY DE CREACION').then(pl.lit('LEY DE CREACION'))
        .otherwise(pl.lit('NO LICENCIADO'))
        .alias('ESTADO_LICENCIAMIENTO')
    )
    .drop('LICENCIA')
    .with_row_index('SK_Universidad', offset=1)
)
dim_univ.write_parquet(GOLD / 'dim_universidad.parquet')
print(f'DimUniversidad: {dim_univ.height:,} filas → dim_universidad.parquet')

# --- DimPrograma: unión ING + MAT ---
cols_prog = ['CODIGO_SIU_PROGRAMA', 'NOMBRE_PROGRAMA', 'CODIGO_GRUPO_1', 'NOMBRE_GRUPO_1', 'CODIGO_GRUPO_3', 'NOMBRE_GRUPO_3', 'NIVEL_ACADEMICO']
dim_prog = (
    pl.concat([
        pl.scan_parquet(ING_V2).select(cols_prog).unique().collect(),
        pl.scan_parquet(MAT_V2).select(cols_prog).unique().collect(),
    ])
    .unique()
    .unique(subset=['CODIGO_SIU_PROGRAMA'])
    .sort('CODIGO_SIU_PROGRAMA')
    .with_row_index('SK_Programa', offset=1)
)
dim_prog.write_parquet(GOLD / 'dim_programa.parquet')
print(f'DimPrograma: {dim_prog.height:,} filas → dim_programa.parquet')

# --- DimPeriodo: semestres (MAT) + anuales (ING, SEMESTRE=NULL) ---
dim_periodo_sem = (
    pl.scan_parquet(MAT_V2).select('PERIODO_ESTANDARIZADO').unique().collect()
    .with_columns([
        pl.col('PERIODO_ESTANDARIZADO').str.slice(0, 4).cast(pl.Int64).alias('ANIO'),
        pl.col('PERIODO_ESTANDARIZADO').str.slice(5, 1).cast(pl.Int64).alias('SEMESTRE'),
        pl.lit('SEMESTRAL').alias('TIPO_PERIODO'),
    ])
    .select(['ANIO', 'SEMESTRE', pl.col('PERIODO_ESTANDARIZADO').alias('LABEL_PERIODO'), 'TIPO_PERIODO'])
)
dim_periodo_an = (
    pl.scan_parquet(ING_V2).select('PROCESO_ESTANDARIZADO').unique().collect()
    .with_columns([
        pl.col('PROCESO_ESTANDARIZADO').cast(pl.Int64).alias('ANIO'),
        pl.lit(None, dtype=pl.Int64).alias('SEMESTRE'),
        pl.col('PROCESO_ESTANDARIZADO').cast(pl.String).alias('LABEL_PERIODO'),
        pl.lit('ANUAL').alias('TIPO_PERIODO'),
    ])
    .select(['ANIO', 'SEMESTRE', 'LABEL_PERIODO', 'TIPO_PERIODO'])
)
dim_periodo = (
    pl.concat([dim_periodo_sem, dim_periodo_an])
    .unique(subset=['ANIO', 'SEMESTRE'])
    .sort('ANIO', 'SEMESTRE')
    .with_row_index('SK_Periodo', offset=1)
)
dim_periodo.write_parquet(GOLD / 'dim_periodo.parquet')
print(f'DimPeriodo: {dim_periodo.height:,} filas → dim_periodo.parquet')

# --- DimUbicacion: DEPARTAMENTO + PROVINCIA (FILIAL en ING, LOCAL en MAT) + Region_Sur ---
dim_ubicacion = (
    pl.concat([
        pl.scan_parquet(ING_V2).select([
            pl.col('DEPARTAMENTO_FILIAL').alias('DEPARTAMENTO'),
            pl.col('PROVINCIA_FILIAL').alias('PROVINCIA'),
        ]).unique().collect(),
        pl.scan_parquet(MAT_V2).select([
            pl.col('DEPARTAMENTO_LOCAL').alias('DEPARTAMENTO'),
            pl.col('PROVINCIA_LOCAL').alias('PROVINCIA'),
        ]).unique().collect(),
    ])
    .unique()
    .unique(subset=['DEPARTAMENTO', 'PROVINCIA'])
    .sort('DEPARTAMENTO', 'PROVINCIA')
    .with_columns(pl.col('DEPARTAMENTO').is_in(DEPARTAMENTOS_SUR).alias('Region_Sur'))
    .with_row_index('SK_Ubicacion', offset=1)
)
dim_ubicacion.write_parquet(GOLD / 'dim_ubicacion.parquet')
print(f'DimUbicacion: {dim_ubicacion.height:,} filas → dim_ubicacion.parquet')

# --- DimLocal: solo Matriculados (se escribe para que 07b la recargue) ---
dim_local = (
    pl.scan_parquet(MAT_V2)
    .select(['CODIGO_LOCAL', 'DEPARTAMENTO_LOCAL', 'PROVINCIA_LOCAL', 'DISTRITO_LOCAL', 'ES_LOCAL_PRINCIPAL', 'CODIGO_UBIGEO_INEI_LOCAL'])
    .unique()
    .collect()
    .unique(subset=['CODIGO_LOCAL'])
    .sort('CODIGO_LOCAL')
    .with_row_index('SK_Local', offset=1)
)
dim_local.write_parquet(GOLD / 'dim_local.parquet')
print(f'DimLocal: {dim_local.height:,} filas → dim_local.parquet')
print()
print('Resumen de dimensiones generadas:')
for f in sorted(GOLD.glob('dim_*.parquet')):
    d = pl.scan_parquet(f)
    print(f'  {f.name:<26} {d.select(pl.len()).collect().item():>7,} filas')

DimUniversidad: 177 filas → dim_universidad.parquet


DimPrograma: 537 filas → dim_programa.parquet


DimPeriodo: 18 filas → dim_periodo.parquet


DimUbicacion: 98 filas → dim_ubicacion.parquet


DimLocal: 65 filas → dim_local.parquet

Resumen de dimensiones generadas:
  dim_local.parquet               65 filas
  dim_periodo.parquet             18 filas
  dim_programa.parquet           537 filas
  dim_ubicacion.parquet           98 filas
  dim_universidad.parquet        177 filas


---
## SECCIÓN 2 · Construcción de `fact_ingresantes.parquet`

Se carga `ingresantes_clean_v2.parquet` completo (`pl.read_parquet`, cabe en RAM) y se sustituyen las **claves naturales** por **FK** hacia las dimensiones (joins contra los `SK_*`). Se conservan los atributos degenerados (`SEXO`, `EDAD`, `NACIONALIDAD`, `Region_Sur`) y `GUID_PERSONA` como medida de conteo.

In [3]:
# --- FactIngresantes: claves → FK (dataset pequeño, en memoria) ---
ing = pl.read_parquet(ING_V2)

fact_ing = (
    ing.lazy()
    .join(dim_univ.select(['CODIGO_INEI', 'SK_Universidad']).lazy(), on='CODIGO_INEI', how='left')
    .join(dim_prog.select(['CODIGO_SIU_PROGRAMA', 'SK_Programa']).lazy(), on='CODIGO_SIU_PROGRAMA', how='left')
    .join(
        dim_periodo.filter(pl.col('SEMESTRE').is_null()).select(['ANIO', 'SK_Periodo']).lazy(),
        left_on=pl.col('PROCESO_ESTANDARIZADO').cast(pl.Int64),
        right_on='ANIO',
        how='left',
    )
    .join(
        dim_ubicacion.select(['DEPARTAMENTO', 'PROVINCIA', 'SK_Ubicacion']).lazy(),
        left_on=['DEPARTAMENTO_FILIAL', 'PROVINCIA_FILIAL'],
        right_on=['DEPARTAMENTO', 'PROVINCIA'],
        how='left',
    )
    .select([
        'SK_Universidad', 'SK_Programa', 'SK_Periodo', 'SK_Ubicacion',
        'GUID_PERSONA', 'SEXO', 'EDAD', 'NACIONALIDAD', 'Region_Sur',
    ])
    .rename({
        'SK_Universidad': 'FK_Universidad',
        'SK_Programa': 'FK_Programa',
        'SK_Periodo': 'FK_Periodo',
        'SK_Ubicacion': 'FK_Ubicacion',
    })
    .collect()
)
fact_ing.write_parquet(GOLD / 'fact_ingresantes.parquet')
print(f'FactIngresantes: {fact_ing.shape} → fact_ingresantes.parquet')
print(fact_ing.head(3))

FactIngresantes: (3119994, 9) → fact_ingresantes.parquet
shape: (3, 9)
┌────────────┬────────────┬────────────┬────────────┬───┬───────────┬──────┬───────────┬───────────┐
│ FK_Univers ┆ FK_Program ┆ FK_Periodo ┆ FK_Ubicaci ┆ … ┆ SEXO      ┆ EDAD ┆ NACIONALI ┆ Region_Su │
│ idad       ┆ a          ┆ ---        ┆ on         ┆   ┆ ---       ┆ ---  ┆ DAD       ┆ r         │
│ ---        ┆ ---        ┆ u32        ┆ ---        ┆   ┆ str       ┆ str  ┆ ---       ┆ ---       │
│ u32        ┆ u32        ┆            ┆ u32        ┆   ┆           ┆      ┆ str       ┆ bool      │
╞════════════╪════════════╪════════════╪════════════╪═══╪═══════════╪══════╪═══════════╪═══════════╡
│ 116        ┆ 63         ┆ 1          ┆ 64         ┆ … ┆ FEMENINO  ┆ 38   ┆ PERUANA   ┆ false     │
│ 116        ┆ 59         ┆ 1          ┆ 64         ┆ … ┆ FEMENINO  ┆ 24   ┆ PERUANA   ┆ false     │
│ 116        ┆ 59         ┆ 1          ┆ 64         ┆ … ┆ MASCULINO ┆ 48   ┆ VENEZOLAN ┆ false     │
│            ┆      

---
## SECCIÓN 3 · Validación rápida y liberación de memoria

Se verifica que `fact_ingresantes` tiene el **mismo número de filas** que la fuente V2 y, al final, se **libera toda la memoria** (`del` + `gc.collect()`) para dejar el entorno limpio antes del notebook 07b (Matriculados).

In [4]:
print('VALIDACIÓN RÁPIDA')
print('=' * 72)
ing_n = pl.scan_parquet(ING_V2).select(pl.len()).collect().item()
fact_ing_n = pl.scan_parquet(GOLD / 'fact_ingresantes.parquet').select(pl.len()).collect().item()
print(f'FactIngresantes: {fact_ing_n:,} filas')
print(f'Ingresantes V2:  {ing_n:,} filas')
print(f'Coinciden:       {fact_ing_n == ing_n}')
assert fact_ing_n == ing_n, 'El número de filas del hecho no coincide con la fuente V2'
print('=' * 72)
print()

# Liberación completa de memoria (Matriculados se procesará en 07b, recargando las dimensiones desde disco)
del ing, fact_ing, dim_univ, dim_prog, dim_periodo, dim_ubicacion, dim_local
gc.collect()
print(f'RSS final: {rss_actual_gb():.2f} GB')
print('OK: Gold de Ingresantes generado. Dimensiones en disco listas para 07b (Matriculados).')

VALIDACIÓN RÁPIDA
FactIngresantes: 3,119,994 filas
Ingresantes V2:  3,119,994 filas
Coinciden:       True

RSS final: 2.31 GB
OK: Gold de Ingresantes generado. Dimensiones en disco listas para 07b (Matriculados).
